Variables consideradas en el análisis

| Variable | Tipo | Unidad | Descripción | Rol en el análisis |
| :-- | :-- | :-- | :-- | :-- |
| FECHA | Temporal | Fecha y hora | Fecha y hora del registro horario de emisiones. | Índice temporal de la serie |
| CONCENTRACION_NOX_MG_NM3 | Continua | mg/Nm³ | Concentración de óxidos de nitrógeno, NOX, corregida por O₂ en base seca. | Variable objetivo |
| CONCENTRACION_NOX_PPM | Continua | ppm | Concentración de NOX medida en partes por millón. | Variable complementaria |
| POTENCIA_BRUTA_MWH | Continua | MWh | Potencia bruta a la cual operó la fuente durante el promedio horario registrado. | Variable explicativa opcional |
| OXIGENO_PORCENTAJE_BASE_SECA | Continua | % | Concentración de oxígeno en porcentaje y base seca. | Variable explicativa opcional |
| FLUJO_GASES_SALIDA_NM3_H | Continua | Nm³/h | Flujo de gases de salida en base seca. | Variable explicativa opcional |
| TEMPERATURA_GASES_SALIDA_C | Continua | °C | Temperatura de gases de salida. | Variable explicativa opcional |
| HUMEDAD_PORCENTAJE | Continua | % | Humedad expresada como porcentaje de H₂O. | Variable explicativa opcional |
| TIPO_DATO_NOX | Categórica | Sin unidad | Código que identifica el tipo de dato registrado para NOX. | Control de calidad / clasificación del dato |
| NombreCentral | Categórica | Sin unidad | Nombre de la central termoeléctrica registrada. | Variable de filtro |
| UGE | Categórica | Sin unidad | Unidad generadora. | Variable de filtro |
| archivo_origen | Categórica | Sin unidad | Archivo desde el cual proviene cada registro. | Control de trazabilidad |
| periodo_archivo | Categórica | Año-trimestre | Período reportado en el nombre del archivo. | Control de trazabilidad |

In [52]:
import pandas as pd
from pathlib import Path
import unicodedata
import re
import csv

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

CARPETA_DATOS = Path("datos")
CARPETA_SALIDAS = Path("salidas")

CARPETA_SALIDAS.mkdir(exist_ok=True)

ENCODING = "utf-16"
SEPARADOR = ";"
DECIMAL = "."

CENTRAL_OBJETIVO = "ANGAMOS"
UNIDAD_OBJETIVO = "ANGAMOS 1"

COLUMNA_CENTRAL = "NombreCentral"
COLUMNA_UNIDAD = "UGE"
COLUMNA_FECHA = "FECHA"
COLUMNA_TIPO_DATO_NOX = "TIPO_DATO_NOX"

VARIABLE_SERIE = "CONCENTRACION_NOX_MG_NM3"

ANIOS_REVISION = [2022,2023,2024,2025]

# ============================================================
# FUNCIONES
# ============================================================

def normalizar_texto(valor):
    if pd.isna(valor):
        return ""

    texto = str(valor).strip()

    texto = texto.replace('"', "")
    texto = texto.replace("'", "")
    texto = texto.strip().upper()
    texto = " ".join(texto.split())

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caracter for caracter in texto
        if not unicodedata.combining(caracter)
    )

    return texto


def extraer_periodo_archivo(nombre_archivo):
    """
    Extrae el período desde nombres tipo:
    PH2017-3_Act2018-11-14.csv

    Retorna:
    2017-3
    """
    patron = r"PH(\d{4}-\d)_Act"
    coincidencia = re.search(patron, nombre_archivo)

    if coincidencia:
        return coincidencia.group(1)

    return None


def leer_archivos_ph(carpeta):
    archivos = sorted([
        archivo for archivo in carpeta.iterdir()
        if archivo.is_file()
        and archivo.name.upper().startswith("PH")
        and archivo.suffix.upper() == ".CSV"
    ])

    if not archivos:
        raise FileNotFoundError(
            "No se encontraron archivos PH*.csv en la carpeta datos/"
        )

    bases = []
    archivos_con_error = []

    codificaciones_a_probar = [
        "utf-16",
        "utf-16-le",
        "utf-8-sig",
        "utf-8",
        "latin-1"
    ]

    columnas_minimas = [
        COLUMNA_CENTRAL,
        COLUMNA_UNIDAD,
        COLUMNA_FECHA,
        COLUMNA_TIPO_DATO_NOX
    ]

    for archivo in archivos:
        archivo_leido = False
        ultimo_error = None

        for encoding_prueba in codificaciones_a_probar:
            try:
                df_temp = pd.read_csv(
                    archivo,
                    sep=SEPARADOR,
                    encoding=encoding_prueba,
                    dtype=str,
                    decimal=DECIMAL,
                    engine="python",
                    quoting=csv.QUOTE_NONE,
                    on_bad_lines="warn"
                )

                df_temp.columns = (
                    df_temp.columns
                    .str.replace("\ufeff", "", regex=False)
                    .str.replace('"', "", regex=False)
                    .str.replace("'", "", regex=False)
                    .str.strip()
                )

                columnas_faltantes = [
                    col for col in columnas_minimas
                    if col not in df_temp.columns
                ]

                if columnas_faltantes:
                    continue

                df_temp["archivo_origen"] = archivo.name
                df_temp["periodo_archivo"] = extraer_periodo_archivo(archivo.name)
                df_temp["encoding_usado"] = encoding_prueba

                bases.append(df_temp)

                print(f"Archivo leído correctamente: {archivo.name}")

                archivo_leido = True
                break

            except Exception as e:
                ultimo_error = str(e)

        if not archivo_leido:
            print(f"No se pudo leer correctamente: {archivo.name}")

            archivos_con_error.append({
                "archivo": archivo.name,
                "error": ultimo_error
            })

    if not bases:
        raise ValueError(
            "No se pudo leer ningún archivo PH*.csv correctamente."
        )

    df_total = pd.concat(bases, ignore_index=True)

    if archivos_con_error:
        pd.DataFrame(archivos_con_error).to_csv(
            CARPETA_SALIDAS / "00_archivos_con_error_lectura.csv",
            index=False,
            encoding="utf-8-sig"
        )

    resumen_encoding = (
        df_total[["archivo_origen", "encoding_usado"]]
        .drop_duplicates()
        .sort_values("archivo_origen")
    )

    resumen_encoding.to_csv(
        CARPETA_SALIDAS / "00_resumen_encoding_usado.csv",
        index=False,
        encoding="utf-8-sig"
    )

    resumen_registros = (
        df_total
        .groupby("archivo_origen")
        .size()
        .reset_index(name="registros_leidos")
        .sort_values("archivo_origen")
    )

    resumen_registros.to_csv(
        CARPETA_SALIDAS / "00_resumen_registros_leidos_por_archivo.csv",
        index=False,
        encoding="utf-8-sig"
    )

    return df_total


def validar_columnas(df, columnas_requeridas):
    """
    Verifica que las columnas necesarias existan en la base.
    """
    faltantes = [
        columna for columna in columnas_requeridas
        if columna not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan columnas necesarias en la base: {faltantes}"
        )


def convertir_a_numerico(serie):
    serie_limpia = (
        serie.astype(str)
        .str.replace('"', "", regex=False)
        .str.replace("'", "", regex=False)
        .str.strip()
    )

    return pd.to_numeric(serie_limpia, errors="coerce")


Leer los archivos

In [53]:
df = leer_archivos_ph(CARPETA_DATOS)

print("\nCantidad total de registros leídos:")
print(len(df))

print("\nCantidad total de columnas:")
print(len(df.columns))

print("\nColumnas encontradas:")
print(df.columns.tolist())

Archivo leído correctamente: PH2022-1_Act2024-03-11.csv
Archivo leído correctamente: PH2022-2_Act2024-03-11.csv
Archivo leído correctamente: PH2022-3_Act2024-03-11.csv
Archivo leído correctamente: PH2022-4_Act2024-03-11.csv
Archivo leído correctamente: PH2023-1_Act2024-03-11.csv
Archivo leído correctamente: PH2023-2_Act2024-03-11.csv
Archivo leído correctamente: PH2023-3_Act2024-03-11.csv
Archivo leído correctamente: PH2023-4_Act2024-03-11.csv
Archivo leído correctamente: PH2024-1_Act2025-05-28.csv
Archivo leído correctamente: PH2024-2_Act2025-05-28.csv
Archivo leído correctamente: PH2024-3_Act2025-05-28.csv
Archivo leído correctamente: PH2024-4_Act2025-05-28.csv
Archivo leído correctamente: PH2025-1_Act2025-12-09.csv
Archivo leído correctamente: PH2025-2_Act2025-12-09.csv
Archivo leído correctamente: PH2025-3_Act2025-12-09.csv

Cantidad total de registros leídos:
2784702

Cantidad total de columnas:
45

Columnas encontradas:
['Fila', 'NombreCentral', 'Chimenea', 'UGE', 'Periodo', 'Fec

Validación de columnas

In [46]:
columnas_necesarias = [
    COLUMNA_CENTRAL,
    COLUMNA_UNIDAD,
    COLUMNA_FECHA,
    COLUMNA_TIPO_DATO_NOX
]

validar_columnas(df, columnas_necesarias)

print("Todas las columnas necesarias están presentes.")

Todas las columnas necesarias están presentes.


Filtro ANGAMOS / ANGAMOS 1

In [57]:
df["central_norm"] = df[COLUMNA_CENTRAL].apply(normalizar_texto)
df["unidad_norm"] = df[COLUMNA_UNIDAD].apply(normalizar_texto)

central_objetivo_norm = normalizar_texto(CENTRAL_OBJETIVO)
unidad_objetivo_norm = normalizar_texto(UNIDAD_OBJETIVO)

df_angamos = df[
    (df["central_norm"] == central_objetivo_norm) &
    (df["unidad_norm"] == unidad_objetivo_norm)
].copy()

print("Registros filtrados para ANGAMOS / ANGAMOS 1:")
print(len(df_angamos))

df_angamos.to_csv(
    CARPETA_SALIDAS / "01_base_filtrada_angamos_1.csv",
    index=False,
    encoding="utf-8-sig"
)

if df_angamos.empty:
    raise ValueError(
        "No se encontraron registros para ANGAMOS / ANGAMOS 1. "
        "Revisa si los nombres están escritos de otra forma en la base."
    )


Registros filtrados para ANGAMOS / ANGAMOS 1:
32856


Conversión y validación de fecha


In [55]:
df_angamos[COLUMNA_FECHA_ORIGINAL := "FECHA_ORIGINAL"] = df_angamos[COLUMNA_FECHA]

df_angamos[COLUMNA_FECHA] = pd.to_datetime(
    df_angamos[COLUMNA_FECHA],
    errors="coerce",
    dayfirst=True
)

fechas_invalidas = df_angamos[df_angamos[COLUMNA_FECHA].isna()].copy()

print("Registros con FECHA inválida o no interpretable:")
print(len(fechas_invalidas))

fechas_invalidas.to_csv(
    CARPETA_SALIDAS / "02_fechas_invalidas.csv",
    index=False,
    encoding="utf-8-sig"
)

df_angamos_valido = df_angamos.dropna(subset=[COLUMNA_FECHA]).copy()

df_angamos_valido["anio"] = df_angamos_valido[COLUMNA_FECHA].dt.year
df_angamos_valido["mes"] = df_angamos_valido[COLUMNA_FECHA].dt.month
df_angamos_valido["dia"] = df_angamos_valido[COLUMNA_FECHA].dt.date
df_angamos_valido["hora"] = df_angamos_valido[COLUMNA_FECHA].dt.hour

df_angamos_valido = df_angamos_valido.sort_values(COLUMNA_FECHA)

Registros con FECHA inválida o no interpretable:
19896


Disponibilidad de datos por año

In [56]:
resumen_anios = (
    df_angamos_valido
    .groupby("anio")
    .size()
    .reset_index(name="cantidad_registros")
    .sort_values("anio")
)

print("\nResumen de registros por año:")
print(resumen_anios)

resumen_anios.to_csv(
    CARPETA_SALIDAS / "03_resumen_registros_por_anio.csv",
    index=False,
    encoding="utf-8-sig"
)

for anio in ANIOS_REVISION:
    datos_anio = df_angamos_valido[df_angamos_valido["anio"] == anio].copy()

    print(f"\n¿Existen datos para {anio}?")
    print("Sí" if not datos_anio.empty else "No")

    datos_anio.to_csv(
        CARPETA_SALIDAS / f"04_datos_angamos_1_{anio}.csv",
        index=False,
        encoding="utf-8-sig"
    )


Resumen de registros por año:
   anio  cantidad_registros
0  2022                3456
1  2023                3456
2  2024                3456
3  2025                2592

¿Existen datos para 2022?
Sí

¿Existen datos para 2023?
Sí

¿Existen datos para 2024?
Sí

¿Existen datos para 2025?
Sí


Duplicados por fecha y hora

In [58]:
duplicados_fecha_hora = df_angamos_valido[
    df_angamos_valido.duplicated(subset=[COLUMNA_FECHA], keep=False)
].copy()

resumen_duplicados = (
    duplicados_fecha_hora
    .groupby(COLUMNA_FECHA)
    .size()
    .reset_index(name="cantidad_registros_misma_fecha_hora")
    .sort_values(COLUMNA_FECHA)
)

print("\nCantidad de registros involucrados en duplicados:")
print(len(duplicados_fecha_hora))

print("\nCantidad de fechas-horas duplicadas:")
print(len(resumen_duplicados))

duplicados_fecha_hora.to_csv(
    CARPETA_SALIDAS / "05_duplicados_por_fecha_hora_detalle.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_duplicados.to_csv(
    CARPETA_SALIDAS / "06_duplicados_por_fecha_hora_resumen.csv",
    index=False,
    encoding="utf-8-sig"
)

for anio in ANIOS_REVISION:
    datos_anio = df_angamos_valido[
        df_angamos_valido["anio"] == anio
    ].copy()

    duplicados_anio = datos_anio[
        datos_anio.duplicated(subset=[COLUMNA_FECHA], keep=False)
    ].copy()

    resumen_dup_anio = (
        duplicados_anio
        .groupby(COLUMNA_FECHA)
        .size()
        .reset_index(name="cantidad_registros_misma_fecha_hora")
        .sort_values(COLUMNA_FECHA)
    )

    print(f"\nDuplicados en {anio}:")
    print("Registros involucrados:", len(duplicados_anio))
    print("Fechas-horas duplicadas:", len(resumen_dup_anio))

    duplicados_anio.to_csv(
        CARPETA_SALIDAS / f"07_duplicados_detalle_{anio}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    resumen_dup_anio.to_csv(
        CARPETA_SALIDAS / f"08_duplicados_resumen_{anio}.csv",
        index=False,
        encoding="utf-8-sig"
    )



Cantidad de registros involucrados en duplicados:
0

Cantidad de fechas-horas duplicadas:
0

Duplicados en 2022:
Registros involucrados: 0
Fechas-horas duplicadas: 0

Duplicados en 2023:
Registros involucrados: 0
Fechas-horas duplicadas: 0

Duplicados en 2024:
Registros involucrados: 0
Fechas-horas duplicadas: 0

Duplicados en 2025:
Registros involucrados: 0
Fechas-horas duplicadas: 0


Análisis de TIPO_DATO_NOX




In [59]:
df_angamos_valido["TIPO_DATO_NOX_norm"] = (
    df_angamos_valido[COLUMNA_TIPO_DATO_NOX]
    .apply(normalizar_texto)
)

resumen_tipo_nox_global = (
    df_angamos_valido
    .groupby("TIPO_DATO_NOX_norm", dropna=False)
    .size()
    .reset_index(name="cantidad_registros")
    .sort_values("cantidad_registros", ascending=False)
)

print("\nResumen global de TIPO_DATO_NOX:")
print(resumen_tipo_nox_global)

resumen_tipo_nox_global.to_csv(
    CARPETA_SALIDAS / "09_resumen_tipo_dato_nox_global.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_tipo_nox_anio = (
    df_angamos_valido
    .groupby(["anio", "TIPO_DATO_NOX_norm"], dropna=False)
    .size()
    .reset_index(name="cantidad_registros")
    .sort_values(["anio", "cantidad_registros"], ascending=[True, False])
)

print("\nResumen de TIPO_DATO_NOX por año:")
print(resumen_tipo_nox_anio)

resumen_tipo_nox_anio.to_csv(
    CARPETA_SALIDAS / "10_resumen_tipo_dato_nox_por_anio.csv",
    index=False,
    encoding="utf-8-sig"
)

for anio in ANIOS_REVISION:
    resumen_tipo_nox_anio_filtrado = resumen_tipo_nox_anio[
        resumen_tipo_nox_anio["anio"] == anio
    ].copy()

    resumen_tipo_nox_anio_filtrado.to_csv(
        CARPETA_SALIDAS / f"11_resumen_tipo_dato_nox_{anio}.csv",
        index=False,
        encoding="utf-8-sig"
    )


Resumen global de TIPO_DATO_NOX:
  TIPO_DATO_NOX_norm  cantidad_registros
0                 DM               12647
1                 DS                 313

Resumen de TIPO_DATO_NOX por año:
   anio TIPO_DATO_NOX_norm  cantidad_registros
0  2022                 DM                3314
1  2022                 DS                 142
2  2023                 DM                3395
3  2023                 DS                  61
4  2024                 DM                3420
5  2024                 DS                  36
6  2025                 DM                2518
7  2025                 DS                  74


Preparación de la variable continua para la serie de tiempo

In [61]:
columnas_nox = [
    "CONCENTRACION_NOX_PPM",
    "CONCENTRACION_NOX_MG_NM3",
    "CONCENTRACION_NOX_MG_MWH",
    "TIPO_DATO_NOX"
]

columnas_nox_existentes = [
    col for col in columnas_nox
    if col in df_angamos_valido.columns
]

muestra_nox = df_angamos_valido[
    columnas_nox_existentes + [COLUMNA_FECHA, "archivo_origen"]
].head(30)

print(muestra_nox)

muestra_nox.to_csv(
    CARPETA_SALIDAS / "diagnostico_muestra_valores_nox.csv",
    index=False,
    encoding="utf-8-sig"
)

     CONCENTRACION_NOX_PPM  CONCENTRACION_NOX_MG_NM3 CONCENTRACION_NOX_MG_MWH  \
4321    181,34500122070312                       NaN              1060032,625   
4323       178,27099609375                       NaN               1078042,25   
4325    180,44200134277344                       NaN              1116326,375   
4327    181,22300720214844                       NaN                1128125,0   
4329    176,66900634765625                       NaN                1046464,5   
4331    173,93899536132812                       NaN             1040274,5625   
4333    184,52699279785156                       NaN                1140778,0   
4335    179,95399475097656                       NaN                1293484,5   
4337    160,14300537109375                       NaN                1398808,0   
4339    141,33299255371094                       NaN              1368821,375   
4341    152,64300537109375                       NaN              2041485,375   
4343    140,88699340820312  

In [60]:
if VARIABLE_SERIE not in df_angamos_valido.columns:
    raise ValueError(
        f"La variable {VARIABLE_SERIE} no existe en la base. "
        "Revisa el nombre de la columna o selecciona otra variable continua."
    )

df_angamos_valido[VARIABLE_SERIE] = convertir_a_numerico(
    df_angamos_valido[VARIABLE_SERIE]
)

serie_tiempo = df_angamos_valido[
    [
        COLUMNA_FECHA,
        VARIABLE_SERIE,
        COLUMNA_TIPO_DATO_NOX,
        "TIPO_DATO_NOX_norm",
        "anio",
        "mes",
        "dia",
        "hora",
        "archivo_origen",
        "periodo_archivo"
    ]
].copy()

serie_tiempo = serie_tiempo.sort_values(COLUMNA_FECHA)

serie_tiempo.to_csv(
    CARPETA_SALIDAS / "12_serie_tiempo_nox_mg_nm3.csv",
    index=False,
    encoding="utf-8-sig"
)

registros_variable_valida = serie_tiempo[VARIABLE_SERIE].notna().sum()
registros_variable_faltante = serie_tiempo[VARIABLE_SERIE].isna().sum()

print(f"\nVariable seleccionada para serie de tiempo: {VARIABLE_SERIE}")
print("Registros con valor válido:", registros_variable_valida)
print("Registros con valor faltante:", registros_variable_faltante)


Variable seleccionada para serie de tiempo: CONCENTRACION_NOX_MG_NM3
Registros con valor válido: 5
Registros con valor faltante: 12955
